# Pattern C · Cross App Access, machine to machine

There's no person. An app (the event source) gets its own token, and the event agent acts **for that app**: the partner's token names the app as `sub` and the agent as `act`.

| Step | Who asks | Okta app / AI agent | Gets | From | Grant |
|---|---|---|---|---|---|
| 1 | the event source | `anyco-event-source-app` | T1 | `anyco-event-agent-as` | client credentials, `private_key_jwt` (KMS) |
| 2a | the event agent | `anyco-event-agent-oai` | ID-JAG | Okta org server | token exchange, subject T1 (machine access) |
| 2b | the event agent | `anyco-event-agent-oai` | partner token | `anyco-partner-mcp-as` | JWT bearer, assertion = the ID-JAG |

Step 1 is plain `requests` (in the demo, the trigger Lambda does it). Step 2 uses the event agent's `@requires_access_token` decorators and its providers from `deploy.py` (`anyco-event-step1`, `anyco-event-step2`).

**Signing keys:** both clients sign with AWS KMS keys whose policies allow only their own roles (the trigger Lambda, the event agent). To run this notebook as yourself, the next cell lets the console operator sign with those two keys too; the last cell puts the policies back.

In [ ]:
import asyncio, base64, hashlib, json, os, secrets, sys, time, urllib.parse, uuid
sys.path.insert(0, "..")                       # the repo root: config.py reads ../.env and ../state.json
import requests
import config as C

os.environ["AWS_REGION"] = C.REGION            # the decorator builds its own client from AWS_REGION
if C.AWS_PROFILE:
    os.environ["AWS_PROFILE"] = C.AWS_PROFILE
from bedrock_agentcore.identity.auth import requires_access_token
from bedrock_agentcore.runtime.context import BedrockAgentCoreContext
from bedrock_agentcore.services.identity import IdentityClient

E, st = C.os.environ, C.state()
ac = C.session().client("bedrock-agentcore-control")   # to create the notebook's providers
identity = IdentityClient(C.REGION)                     # to bind a JWT to the workload identity

# An older boto3 doesn't know the KMS-signed client assertion (privateKeyJwtConfig) the providers use.
_fields = ac.meta.service_model.shape_for("CustomOauth2ProviderConfigInput").members
if "privateKeyJwtConfig" not in _fields:
    import boto3
    raise RuntimeError(f"This kernel's boto3 {boto3.__version__} is too old ({sys.executable}). "
                       "Run the notebook on the repo's .venv kernel, or: pip install -r notebooks/requirements.txt")

def claims(jwt):
    return json.loads(base64.urlsafe_b64decode(jwt.split(".")[1] + "=="))

def show(name, jwt):
    """A bearer token: the raw value shortened, then the header and every claim. The signature is shown
    only as a length, so nothing printed here is a usable credential."""
    h, p, sig = jwt.split(".")
    dec = lambda x: json.loads(base64.urlsafe_b64decode(x + "=="))
    print(f"{name}  ·  {len(jwt)} chars  ·  {jwt[:16]}…{jwt[-6:]}")
    print(json.dumps({"header": dec(h), "payload": dec(p), "signature": f"<{len(sig)} chars, not shown>"}, indent=1))

def act_chain(jwt):
    """Every actor in the token, newest first."""
    out, a = [], claims(jwt).get("act")
    while isinstance(a, dict):
        out.append(f"{a['sub']} ({a.get('sub_profile', '')})"); a = a.get("act")
    return out

WORKLOAD = "anyco-notebook-workload"           # a standalone workload identity, to bind each subject token
try:
    ac.create_workload_identity(name=WORKLOAD, tags={"project": "anyco-agentid"})
except ac.exceptions.ValidationException as e:
    if "already exists" not in str(e): raise

def bind(jwt):
    """Make this JWT the subject of the next decorated call: AgentCore Identity returns a workload access
    token bound to it, and the decorators read the one in the context."""
    wat = identity.get_workload_access_token(WORKLOAD, user_token=jwt)["workloadAccessToken"]
    BedrockAgentCoreContext.set_workload_access_token(wat)

def provider(name, cfg):
    """Create or update an AgentCore Identity OAuth2 provider."""
    try:
        ac.create_oauth2_credential_provider(name=name, credentialProviderVendor="CustomOauth2",
                                             oauth2ProviderConfigInput=cfg, tags={"project": "anyco-agentid"})
    except (ac.exceptions.ConflictException, ac.exceptions.ValidationException) as e:
        if "already exists" not in str(e) and not isinstance(e, ac.exceptions.ConflictException): raise
        ac.update_oauth2_credential_provider(name=name, credentialProviderVendor="CustomOauth2", oauth2ProviderConfigInput=cfg)

ACCESS_TOKEN = "urn:ietf:params:oauth:token-type:access_token"
ID_JAG = "urn:ietf:params:oauth:token-type:id-jag"
print("ready ·", C.OKTA_DOMAIN, "·", C.REGION)

## 0 · Let your AWS identity sign with the two Pattern C keys (for this notebook only)

In [ ]:
import kms_keys
for line in kms_keys.apply_policies(operator_test=True):
    print(line)
time.sleep(10)            # key policy changes take a few seconds to apply

## 1 · The event source: T1 by client credentials
The client assertion is a short JWT (`iss` = `sub` = the event source, `aud` = the token URL) that the event source signs with its KMS key. It proves who is asking; **T1 itself comes from Okta**.

In [ ]:
ev = st["event"]
b64 = lambda raw: base64.urlsafe_b64encode(raw).rstrip(b"=").decode()
token_url = f"{ev['agent_issuer']}/v1/token"
now = int(time.time())
head = b64(json.dumps({"alg": "RS256", "typ": "JWT", "kid": ev["source_kid"]}).encode())
body = b64(json.dumps({"iss": ev["source_client_id"], "sub": ev["source_client_id"], "aud": token_url,
                       "iat": now, "exp": now + 300, "jti": str(uuid.uuid4())}).encode())
sig = C.session().client("kms").sign(KeyId=ev["source_kms_key_arn"], Message=hashlib.sha256(f"{head}.{body}".encode()).digest(),
                                     MessageType="DIGEST", SigningAlgorithm="RSASSA_PKCS1_V1_5_SHA_256")["Signature"]
client_assertion = f"{head}.{body}.{b64(sig)}"

r = requests.post(token_url, timeout=20, data={
    "grant_type": "client_credentials", "scope": "agent.invoke", "resource": ev["agent_audience"],
    "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer", "client_assertion": client_assertion})
t1 = r.json()["access_token"]
show("T1 · the event source's own token", t1)

## 2 · The event agent: T1 → ID-JAG → partner token
Machine access lets T1 be the subject: the event source is an allowed caller of `anyco-event-agent-oai`, and T1's audience is the agent's resource URL. The resource connection grants `partner:read` only.

In [ ]:
@requires_access_token(provider_name="anyco-event-step1", scopes=["partner:read"], auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True,
                       custom_parameters={"subject_token_type": ACCESS_TOKEN, "requested_token_type": ID_JAG,
                                          "audience": ev["mcp_issuer"]})
async def event_id_jag(*, access_token):
    return access_token

@requires_access_token(provider_name="anyco-event-step2", scopes=["partner:read"], auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True)
async def partner_token(*, access_token):
    return access_token

bind(t1);       id_jag = await event_id_jag()           # step 2a: subject T1
show("ID-JAG · for anyco-partner-mcp-as", id_jag)
bind(id_jag);   t3 = await partner_token()              # step 2b
show("Partner token · for the partner MCP server", t3)

## 3 · The trail
No user anywhere: `sub` is the event source, and `act` is the event agent acting for it.

In [ ]:
for name, tok in (("T1", t1), ("partner token", t3)):
    c = claims(tok)
    print(f"{name}: aud {c['aud']} · sub {c['sub']} · cid {c['cid']} · scp {c.get('scp')}")
    for i, a in enumerate(act_chain(tok)):
        print(f"      {'act' + '.act' * i}.sub = {a}")

## 4 · Okta limits what's granted
The resource connection allows `partner:read` only, so asking for `partner:write` is refused at step 2a. Okta answers `invalid_scope`; AgentCore Identity reports it as a failed token exchange (HTTP 400).

In [ ]:
@requires_access_token(provider_name="anyco-event-step1", scopes=["partner:write"], auth_flow="ON_BEHALF_OF_TOKEN_EXCHANGE",
                       force_authentication=True,
                       custom_parameters={"subject_token_type": ACCESS_TOKEN, "requested_token_type": ID_JAG,
                                          "audience": ev["mcp_issuer"]})
async def event_id_jag_write(*, access_token):
    return access_token

bind(t1)
try:
    await event_id_jag_write()
except Exception as e:
    print("refused:", str(e)[:300])

## Clean up
Puts the two keys back to one signer each, and removes the workload identity. The deployed demo is untouched.

In [ ]:
for line in kms_keys.apply_policies():
    print(line)
NOTEBOOK_PROVIDERS = []

for name in NOTEBOOK_PROVIDERS:
    try:
        ac.delete_oauth2_credential_provider(name=name); print("deleted provider", name)
    except Exception as e:
        print("skip", name, str(e)[:80])
try:
    ac.delete_workload_identity(name=WORKLOAD); print("deleted workload identity", WORKLOAD)
except Exception as e:
    print("skip", WORKLOAD, str(e)[:80])